In [ ]:
import seaborn as sns
print(sns.__version__)


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os
from glob import glob

In [ ]:
import os
from glob import glob

# Model group order
model_priority = {
    "LGBM_MOC": 0,
    "RF_MOC": 1,
    "LGBM_CC":2,
    "RF_CC": 3,
    "MLRF": 4
}

# Method order (by keyword in file name)
method_priority = {
    "gain": 0,
    "split": 1,
    "gini": 2,
    "entropy": 3,
    "shap": 4,
    "permutation": 5
}

# All matching feature files
csv_files = glob("results/feature/per_model/*.csv")

def extract_sort_key(filename):
    base = os.path.basename(filename)
    base_lower = base.lower()

    # Model: find the full model name matching the file-name prefix
    model = None
    for m in model_priority:
        if base.startswith(m):
            model = m
            break
    model_rank = model_priority.get(model, 99)

    # Method: match keyword
    method_rank = 99
    for method, rank in method_priority.items():
        if method in base_lower:
            method_rank = rank
            break

    return (model_rank, method_rank)

# Sort
csv_files_sorted = sorted(csv_files, key=extract_sort_key)

# Print results
print(f"Found {len(csv_files_sorted)} CSV files sorted by model + method priority:")
for f in csv_files_sorted:
    print("   -", os.path.basename(f))

importance_dict = {}

for filepath in csv_files_sorted :
    filename = os.path.splitext(os.path.basename(filepath))[0].replace("_importance", "")  # e.g. LGBM_MOC_gain
    df = pd.read_csv(filepath)
    df = df.set_index(df.columns[0]).iloc[:20]  
    df = df[~df.index.duplicated(keep='first')]

    importance_dict[filename] = df.iloc[:, 0]  


importance_matrix = pd.DataFrame(importance_dict).fillna(0)

importance_matrix["avg"] = importance_matrix.mean(axis=1)
importance_matrix = importance_matrix.sort_values("avg", ascending=False).drop(columns="avg")

In [ ]:

plt.figure(figsize=(10, 12))
sns.heatmap(
    importance_matrix,
    annot=True,
    fmt=".4f",
    cmap="YlGnBu",
    cbar_kws={"label": "Importance"},
    linewidths=0.3,
    annot_kws={"size": 2}
)
plt.title("Feature Importance Heatmap(classification)", fontsize=14)
plt.xlabel("Model", fontsize=12)
plt.ylabel("Feature", fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(fontsize=8)
plt.tight_layout()


os.makedirs("results/feature/summary", exist_ok=True)
plt.savefig("results/feature/summary/feature_importance_heatmap_all_models_classification.png", dpi=300)
plt.show()

In [ ]:
clean_cols = (
    importance_matrix.columns
    .str.replace("_top20", "", regex=False)
    .str.replace("_features", "", regex=False)
    .str.replace("_importance", "", regex=False)
    .str.strip()
)
importance_matrix.columns = clean_cols
log_matrix = np.log10(importance_matrix + 1e-4)
plt.figure(figsize=(9, 12))
sns.heatmap(
    log_matrix,
    annot=True,
    fmt=".2f",
    cmap="YlGnBu",
    cbar_kws={"label": "Log10(Importance)"},
    linewidths=0.2,
    annot_kws={"size": 3}
)
plt.title("Feature Importance Heatmap (Log-transformed)", fontsize=14)
plt.xlabel("Model", fontsize=12)
plt.ylabel("Feature", fontsize=12)
plt.xticks(rotation=45, ha='right',fontsize=6)
plt.yticks(fontsize=8)
plt.tight_layout()

plt.savefig("results/feature/summary/feature_importance_heatmap_log_transformed_classification.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
from collections import Counter

all_features = []
for series in importance_dict.values():
    all_features.extend(series.index.tolist())


feature_counts = Counter(all_features)
count_df = pd.DataFrame.from_dict(feature_counts, orient="index", columns=["count"])
count_df = count_df.sort_values("count", ascending=False)

In [ ]:
import numpy as np
import matplotlib.cm as cm
import matplotlib.pyplot as plt

#  Normalize counts to [0, 1] for color mapping
counts = count_df["count"].values
norm_counts = (counts - counts.min()) / (counts.max() - counts.min() + 1e-6)
cmap = cm.get_cmap("viridis")  # Alternatives: "plasma", "magma", "cividis"
colors = cmap(norm_counts)

#  Create horizontal bar plot
fig, ax = plt.subplots(figsize=(10, len(count_df) * 0.3))
bars = ax.barh(count_df.index, counts, color=colors, edgecolor='black')

# Add value labels next to each bar
for bar, value in zip(bars, counts):
    ax.text(
        bar.get_width() + 0.1,  # Slightly to the right of the bar
        bar.get_y() + bar.get_height() / 2,
        f"{value}",
        va='center',
        fontsize=8
    )

#  Axis formatting
ax.set_title("Feature Occurrence Count Across All Models", fontsize=14)
ax.set_xlabel("Number of Appearances", fontsize=12)
ax.set_ylabel("Feature", fontsize=12)
ax.invert_yaxis()  # Most frequent feature on top
ax.grid(axis='x', linestyle='--', alpha=0.4)
plt.xticks(fontsize=10)
plt.yticks(fontsize=6)
plt.tight_layout()

#  Save figure
plt.savefig("results/feature/summary/feature_occurrence_barplot_gradient.png", dpi=300)
plt.show()

In [ ]:
from collections import defaultdict
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os
from glob import glob

# 1. Scan all files
csv_files = glob("results/feature/per_model/*.csv")

# 2. Feature count per model group
group_feature_counter = defaultdict(lambda: defaultdict(int))

for filepath in csv_files:
    filename = os.path.splitext(os.path.basename(filepath))[0]
    parts = filename.split("_")

    # Fix: MLRF has no MOC/CC prefix and needs separate handling
    if parts[0] == "MLRF":
        model_group = "MLRF"
    else:
        model_group = "_".join(parts[:2])  # e.g., LGBM_MOC

    df = pd.read_csv(filepath)
    features = df.iloc[:20, 0]  # Assuming first column is the feature name

    for feature in features:
        group_feature_counter[feature][model_group] += 1

# 3. Convert to DataFrame (rows=features, columns=model_groups, values=count)
count_df = pd.DataFrame(group_feature_counter).T.fillna(0).astype(int)

# 3.5: Custom model group order
model_group_order = ["LGBM_MOC", "LGBM_CC", "RF_MOC", "RF_CC", "MLRF"]
model_group_order = [col for col in model_group_order if col in count_df.columns]
count_df = count_df[model_group_order]

# 4. Sort features by total count across all groups
count_df["total"] = count_df.sum(axis=1)
count_df = count_df.sort_values("total", ascending=False).drop(columns="total")

# 5. Plot heatmap
plt.figure(figsize=(9, 12))
sns.heatmap(
    count_df,
    annot=True,
    fmt="d",
    cmap="YlOrBr",
    cbar_kws={"label": "Feature Count"},
    linewidths=0.2,
    annot_kws={"size": 4}
)

plt.title("Feature Appearance Heatmap by Model Group", fontsize=14)
plt.xlabel("Model Group", fontsize=12)
plt.ylabel("Feature", fontsize=12)
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()

# 6. Save image
plt.savefig("results/feature/summary/feature_group_occurrence_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()